# 01 · Transcribir una voz

## Propósito

Convertir sonido en texto con un modelo neuronal ya entrenado, y mirar de cerca
qué se conserva y qué se pierde en esa conversión. No vamos a entrenar nada:
vamos a usar un modelo que ya existe y a auditarlo.

## Qué consume

- Un audio: el que bajaste en el cuaderno 00, o uno propio.
- El modelo Whisper, que se baja solo la primera vez.

## Resultados de aprendizaje

- Explicar qué es ASR y en qué se diferencia entrenar de inferir.
- Transcribir un audio con `faster-whisper` y leer sus segmentos.
- Interpretar los parámetros de la transcripción en vez de aceptarlos.
- Exportar la salida en TXT, SRT y JSON, y decir qué conserva cada uno.
- Encontrar al menos tres errores del modelo escuchando el audio original.

## ◈ Microglosario

| Término | Significado en este laboratorio |
|---|---|
| **Audio** | Archivo sonoro, por ejemplo MP3, WAV, M4A u OGG. Antes de procesarlo, se convierte en una señal numérica. |
| **ASR** | *Automatic Speech Recognition*: reconocimiento automático del habla. Es la tarea de convertir voz en texto. |
| **Whisper** | Familia de modelos neuronales preentrenados para tareas de audio, entre ellas transcripción e identificación de idioma. |
| **Modelo preentrenado** | Modelo cuyos parámetros fueron ajustados anteriormente con muchos ejemplos. En este cuaderno lo usamos; no lo entrenamos. |
| **Inferencia** | Uso de un modelo ya entrenado para generar una salida nueva. Aquí, una transcripción de audio. |
| **Segmento** | Fragmento de audio al que el modelo asigna un inicio, un final y un texto. |
| **Timestamp** | Marca temporal: segundo en el que empieza o termina un segmento. |
| **VAD** | *Voice Activity Detection*: detección de actividad vocal. Sirve para filtrar o separar tramos sin voz. |
| **SRT** | Formato de subtítulos: número de segmento, tiempo inicial, tiempo final y texto. |
| **JSON** | Formato estructurado de datos que conserva los campos de cada segmento. |
| **CPU / GPU** | Procesadores donde se ejecuta el modelo. Una GPU suele acelerar los cálculos de redes neuronales. |
| **Gradio** | Biblioteca de Python para construir interfaces web simples conectadas a funciones de código. |

## Qué es esto que vamos a hacer

```text
archivo de audio
      ↓
señal numérica
      ↓
Whisper: modelo neuronal preentrenado
      ↓
segmentos: inicio + fin + texto
      ↓
TXT / SRT / JSON
```

**La analogía.** Un micrófono no graba palabras: graba cuánto se mueve el aire, miles de
veces por segundo. Lo que queda en un MP3 es una larguísima lista de números que describen
ese movimiento. Ahí no hay ninguna letra.

Lo que hace Whisper es parecido a lo que hacés vos cuando escuchás a alguien hablar en un
bar ruidoso: no oís cada sonido con claridad, oís *lo suficiente*, y completás el resto con
lo que esperás que esa persona esté diciendo. Por eso entendés mal cuando aparece un nombre
propio que no conocías: no lo esperabas.

El modelo hace exactamente eso, y se equivoca exactamente ahí.

**Dónde vive esto hoy.** Los subtítulos automáticos de YouTube, la transcripción de las
notas de voz de WhatsApp, las actas automáticas de Zoom y Meet, los sistemas que transcriben
llamadas de atención al cliente, los subtítulos en vivo de una videollamada. Todo eso es la
misma tarea, y buena parte usa modelos de esta familia.

**Entrenar no es inferir**, y conviene fijarlo antes de seguir.

| | Entrenar | Inferir |
|---|---|---|
| Qué pasa | El modelo ajusta sus parámetros mirando millones de ejemplos | El modelo usa los parámetros que ya tiene |
| Quién lo hizo | OpenAI, con miles de horas de audio | Vos, en esta clase |
| Cuánto cuesta | Semanas de cómputo, mucha plata | Segundos o minutos |
| Se puede cambiar | Sí, pero no es lo que hacemos acá | — |

**En este cuaderno solo inferimos.** El modelo llega hecho, con todo lo que trae adentro:
las voces con las que lo entrenaron, los acentos que escuchó más, el vocabulario que le
resultó frecuente. Nada de eso es neutro, y es la misma discusión del cuaderno `04` de la
unidad anterior.

## ⚙ Configuración

In [ ]:
# faster_whisper: la implementación eficiente de Whisper que vamos a usar.
from faster_whisper import WhisperModel

# time: para medir cuánto tarda cada paso.
import time

# Audio y display: para escuchar el audio dentro del cuaderno.
from IPython.display import Audio, display

print("Librerías cargadas.")

### El modelo y dónde corre

Whisper viene en varios tamaños. **No hay uno mejor para todo:** hay un intercambio entre
calidad, velocidad y memoria disponible.

La celda de abajo elige por vos según el entorno. No hace falta tocarla la primera vez.

In [ ]:
# En una máquina de escritorio lo normal es no tener GPU compatible,
# así que arrancamos por el camino que funciona en todas partes.
NOMBRE_MODELO = "small"
DISPOSITIVO = "cpu"
TIPO_COMPUTO = "int8"

print("Modelo    :", NOMBRE_MODELO)
print("Dispositivo:", DISPOSITIVO)
print("Tipo de cálculo:", TIPO_COMPUTO)
print()
print("Si tenés GPU NVIDIA configurada, podés probar 'large-v3-turbo' con 'cuda'.")

## El audio

Antes de transcribir, **escuchá**. Es el paso que más se saltea y el que más sirve después,
cuando haya que decidir si un error es del modelo o del audio.

Mientras escuchás, registrá:

- ¿Hay una voz o varias? ¿Se pisan?
- ¿Hay ruido, música, eco, distancia al micrófono?
- ¿Se habla rápido? ¿Hay muletillas, risas, cortes?
- ¿Aparecen nombres propios, siglas, palabras en inglés o términos técnicos?

Esas condiciones son parte del problema. El sistema no recibe un texto limpio: recibe una
señal sonora situada, grabada en algún lugar por alguien con algún equipo.

In [ ]:
import os

CARPETA_AUDIOS = "../datos/audios"
os.makedirs(CARPETA_AUDIOS, exist_ok=True)

# Miramos qué hay disponible antes de elegir.
archivos_disponibles = os.listdir(CARPETA_AUDIOS)

assert len(archivos_disponibles) > 0, (
    "No hay audios en datos/audios/. Corré primero el cuaderno 00, "
    "o copiá a mano un MP3 en esa carpeta.")

print("Audios en la carpeta:")
for nombre in archivos_disponibles:
    print("  -", nombre)

De todo lo que hay en la carpeta, hay que elegir uno. Empezamos por el primero de
la lista; si querés otro, cambiá el número o escribí el nombre completo a mano.

**Para la primera prueba, usá un audio de menos de cinco minutos.** Con uno corto podés
escuchar entero y comparar con la transcripción, que es lo que vamos a hacer después. Con
uno de media hora no vas a comparar nada.

In [ ]:
# Poné acá el nombre exacto del archivo que querés transcribir.
AUDIO = CARPETA_AUDIOS + "/" + archivos_disponibles[0]

print("Vamos a transcribir:", AUDIO)

Y acá está el reproductor. **Escuchá ahora**, antes de ver ninguna transcripción.

Si lo escuchás después, ya vas a estar leyendo el texto mientras escuchás, y vas a
entender perfecto cosas que el audio no dice con claridad. Es un efecto conocido y es muy
fuerte: una vez que leíste el subtítulo, ya no podés desoírlo.

In [ ]:
display(Audio(AUDIO))

## Cargar el modelo

Cargar no es entrenar. Acá se baja —la primera vez— y se pone en memoria un conjunto de
parámetros que alguien más ajustó hace tiempo.

**La primera vez tarda**, porque el modelo pesa cientos de megabytes. Las veces siguientes
sale de la caché y es casi instantáneo.

In [ ]:
inicio_de_la_carga = time.perf_counter()

modelo = WhisperModel(
    model_size_or_path=NOMBRE_MODELO,
    device=DISPOSITIVO,
    compute_type=TIPO_COMPUTO,
)

tiempo_de_carga = time.perf_counter() - inicio_de_la_carga
segundos_de_carga = round(tiempo_de_carga, 1)

print("Modelo cargado en", segundos_de_carga, "segundos.")

## Los parámetros de la transcripción

Antes de ejecutar, leé esta tabla. **Los parámetros no son detalles invisibles:** son
decisiones que cambian lo que el sistema produce.

| Parámetro | Valor inicial | Qué controla | Por qué se usa así | Qué probar después |
|---|---|---|---|---|
| `AUDIO` | Archivo elegido | El audio que entra al sistema | Es el material que se transforma en texto | Probar con audios de distintas condiciones sonoras |
| `language` | `'es'` | Idioma esperado | Reduce ambigüedades si sabemos que el audio está en español | Usar `None` para observar la detección automática |
| `vad_filter` | `True` | Filtro de actividad vocal | Ayuda a evitar silencios largos y posibles textos espurios | Comparar con `False` en un audio con pausas |
| `condition_on_previous_text` | `False` | Uso del texto anterior como contexto del siguiente segmento | Reduce el riesgo de propagar un error entre segmentos | Probar `True` en un audio continuo y comparar |
| `beam_size` | `5` | Cantidad de hipótesis exploradas al generar texto | Es un equilibrio razonable entre calidad y tiempo | Probar `1` y `10`; comparar velocidad y cambios |
| `initial_prompt` | No se usa inicialmente | Contexto textual o glosario de dominio | Permite experimentar con términos técnicos o nombres | Agregar solo vocabulario probable del audio |
| `NOMBRE_MODELO` | `small` en CPU / `large-v3-turbo` con GPU | Tamaño y variante del modelo | Equilibra calidad y recursos según el entorno | Comparar `base`, `small` y variantes mayores |
| `DISPOSITIVO` | `cpu` o `cuda` | Dónde se ejecuta el modelo | GPU suele acelerar; CPU permite trabajar sin acelerador | Medir tiempos en ambos, si están disponibles |
| `TIPO_COMPUTO` | `int8` en CPU / `float16` con GPU | Precisión numérica de los cálculos | Reduce memoria y hace viable la ejecución según el hardware | En GPU, probar `int8_float16` si falta memoria |

**Dos ideas que conviene no perder:**

- Un parámetro no es un botón de "mejorar". Cambiarlo puede mejorar un aspecto y empeorar
  otro.
- Para comparar configuraciones, cambiá **una variable por vez** y mantené iguales el audio
  y todo lo demás. Es la misma prueba de pares mínimos que le hiciste a spaCy.

### ✎ Antes de ejecutar

Tu audio dura unos minutos. Antes de correr la celda, escribí tu predicción:

- ¿Cuántos segmentos creés que va a producir? ¿De qué duración cada uno?
- ¿Va a tardar más o menos que la duración del audio?

Ahora la llamada. Prestá atención a lo que **no** pasa: `transcribe()` vuelve
enseguida, pero todavía no transcribió nada.

Devuelve un *generador*, que es una promesa de trabajo: el cálculo recién ocurre cuando
recorrés los segmentos, en la celda siguiente. Es una de esas cosas de Python que confunden
si nadie las señala.

In [ ]:
inicio_del_reloj = time.perf_counter()

generador_de_segmentos, info = modelo.transcribe(
    audio=AUDIO,
    language="es",
    vad_filter=True,
    condition_on_previous_text=False,
    beam_size=5,
)

print("La llamada volvió, pero todavía no se transcribió nada.")

Acá sí trabaja. Recorremos el generador y guardamos cada segmento como un
diccionario con tres campos: cuándo empieza, cuándo termina y qué dice.

Un diccionario por segmento, y una lista con todos: es la misma estructura que usaste para
el corpus del cuaderno 00.

In [ ]:
segmentos = []

for segmento in generador_de_segmentos:
    texto_limpio = segmento.text.strip()
    segmentos.append({
        "inicio": segmento.start,
        "fin": segmento.end,
        "texto": texto_limpio,
    })

tiempo_de_transcripcion = time.perf_counter() - inicio_del_reloj
cantidad_de_segmentos = len(segmentos)

print("Listo. Segmentos producidos:", cantidad_de_segmentos)

Ahora el informe de lo que pasó.

La primera línea es un `assert`: una afirmación que corta el cuaderno si resulta falsa. Si
el audio estaba vacío o el filtro de voz se comió todo, mejor enterarse acá que tres
celdas más abajo.

In [ ]:
assert cantidad_de_segmentos > 0, "No se produjo ningún segmento: revisá el audio."

probabilidad_del_idioma = round(info.language_probability, 3)
segundos_de_proceso = round(tiempo_de_transcripcion, 1)

print("Idioma detectado      :", info.language)
print("Probabilidad del idioma:", probabilidad_del_idioma)
print("Segmentos             :", cantidad_de_segmentos)
print("Tiempo de transcripción:", segundos_de_proceso, "segundos")

**Mirá la probabilidad del idioma.** El modelo no dice "es español": dice "estoy así
de seguro de que es español". Todo lo que hace es de esa naturaleza, incluida cada palabra
que vas a leer abajo. No hay ninguna certeza adentro de esto.

## Leer los segmentos

Cada línea es un tramo del audio con el texto que el modelo le asignó.

In [ ]:
for segmento in segmentos[:12]:
    inicio = round(segmento["inicio"], 2)
    fin = round(segmento["fin"], 2)
    texto = segmento["texto"]
    print("[", inicio, "→", fin, "]", texto)

### ✎ Para pensar

Volvé al reproductor de arriba y **escuchá mientras leés**. Buscá tres fragmentos:

1. Uno que esté bien.
2. Uno dudoso o directamente mal.
3. Uno con un nombre propio, una sigla, una muletilla, una variante de habla o ruido.

Y después la pregunta que importa: **no si hay errores, sino de qué tipo son**. Un número
mal transcripto en un dato económico y una muletilla perdida no tienen la misma
consecuencia. ¿Cuál de tus tres errores sería grave si usaras este texto para algo?

## Tres formas de guardar lo mismo

Vamos a exportar la misma transcripción en tres formatos. No es redundancia: **cada uno
conserva algo distinto y pierde algo distinto.**

| Archivo | Para qué sirve | Qué conserva | Qué pierde |
|---|---|---|---|
| `transcripcion.txt` | Leer, editar, procesar con lo de las unidades 3 y 4 | El texto corrido | Los tiempos y la segmentación |
| `transcripcion.srt` | Subtítulos, revisar sincronía con el audio | Texto y marcas temporales | La estructura de datos |
| `segmentos.json` | Volver a un tramo, reusar en otra aplicación | Inicio, fin y texto de cada segmento | El audio, la prosodia, el contexto |

El JSON es el que no hay que saltear: es el único que permite volver del texto al momento
exacto del audio.

In [ ]:
CARPETA_SALIDAS = "../salidas"

ARCHIVO_TXT = CARPETA_SALIDAS + "/transcripcion.txt"
ARCHIVO_SRT = CARPETA_SALIDAS + "/transcripcion.srt"
ARCHIVO_JSON = CARPETA_SALIDAS + "/segmentos.json"

print("Las salidas van a quedar en:", CARPETA_SALIDAS)

### El TXT

Junta todos los segmentos en un texto corrido. Es lo que vas a poder pasarle a spaCy o a un
vectorizador.

In [ ]:
textos_de_los_segmentos = []

for segmento in segmentos:
    textos_de_los_segmentos.append(segmento["texto"])

texto_completo = " ".join(textos_de_los_segmentos)

with open(ARCHIVO_TXT, "w", encoding="utf-8") as archivo:
    archivo.write(texto_completo)

cantidad_de_caracteres = len(texto_completo)
print("Archivo creado:", ARCHIVO_TXT)
print("Caracteres:", cantidad_de_caracteres)

Miralo. Esto es lo que quedó del audio: un bloque de texto sin tiempos, sin
silencios y sin pausas.

Fijate en la puntuación. El modelo la inventó: nadie dicta comas. Dónde pone un punto y
dónde no es una decisión suya, y esa decisión va a cambiar dónde corta las oraciones
cualquier herramienta que use este texto después.

In [ ]:
primeros_caracteres = texto_completo[:1000]

print(primeros_caracteres)

### El SRT

Un subtítulo por segmento, con sus tiempos. El formato pide las horas así:
`00:01:23,450` — con coma antes de los milisegundos, no con punto.

La función de abajo hace esa conversión. `//` es la división entera y `%` es el resto: los
dos los viste en la unidad 1.

In [ ]:
def tiempo_srt(segundos):
    """Convierte 83.45 en '00:01:23,450', que es lo que pide el formato SRT."""
    horas = int(segundos // 3600)
    minutos = int((segundos % 3600) // 60)
    segundos_enteros = int(segundos % 60)
    milisegundos = int((segundos - int(segundos)) * 1000)
    return f"{horas:02d}:{minutos:02d}:{segundos_enteros:02d},{milisegundos:03d}"


print("Ejemplo:", tiempo_srt(83.45))

Y ahora el archivo. Un bloque por segmento: número, tiempos, texto, y una línea en
blanco que separa.

Llevamos la cuenta con `numero_de_subtitulo`, que arranca en 1 y sube de a uno. Podríamos
pedirle a Python que numere solo, pero con un contador a la vista se entiende exactamente
qué número va a llevar cada bloque.

In [ ]:
numero_de_subtitulo = 1

with open(ARCHIVO_SRT, "w", encoding="utf-8") as archivo:
    for segmento in segmentos:
        momento_inicial = tiempo_srt(segmento["inicio"])
        momento_final = tiempo_srt(segmento["fin"])
        archivo.write(str(numero_de_subtitulo) + "\n")
        archivo.write(momento_inicial + " --> " + momento_final + "\n")
        archivo.write(segmento["texto"] + "\n\n")
        numero_de_subtitulo = numero_de_subtitulo + 1

print("Archivo creado:", ARCHIVO_SRT)

**Probalo de verdad:** abrí el audio en VLC y cargale este `.srt` como pista de
subtítulos. Es la forma más rápida de ver si los tiempos están bien, y suele ser también la
forma más rápida de descubrir que el modelo se comió un tramo.

### El JSON

Conserva la estructura completa: cada segmento con sus tres campos.

`ensure_ascii=False` guarda las tildes como tildes. `indent=2` lo deja legible para una
persona, que en un material de clase importa más que ahorrar bytes.

In [ ]:
import json

with open(ARCHIVO_JSON, "w", encoding="utf-8") as archivo:
    json.dump(segmentos, archivo, ensure_ascii=False, indent=2)

print("Archivo creado:", ARCHIVO_JSON)

Así se ve por dentro. Dos segmentos alcanzan para entender la forma.

In [ ]:
primeros_dos_segmentos = segmentos[:2]
muestra = json.dumps(primeros_dos_segmentos, ensure_ascii=False, indent=2)

print(muestra)

**Los tres son el mismo contenido y ninguno es reemplazable por otro.** El TXT no
sabe cuándo se dijo cada cosa; el SRT no se puede recorrer con Python sin parsearlo; el
JSON no se puede abrir en VLC.

La celda de abajo confirma que los tres existen antes de dar el paso por terminado.

In [ ]:
import os

for ruta in [ARCHIVO_TXT, ARCHIVO_SRT, ARCHIVO_JSON]:
    existe = os.path.exists(ruta)
    assert existe, "Falta generar: " + ruta
    print("OK:", ruta)

print()
print("Los tres archivos quedaron en salidas/. Seguí en el cuaderno 02.")

## Experimento opcional: el glosario de dominio

**Esto no es parte del núcleo del laboratorio.** Es un experimento, y como todo experimento
puede salir mal.

`initial_prompt` le pasa al modelo un texto de contexto antes de empezar. La idea es que si
le avisás que va a escuchar términos técnicos, los reconozca mejor.

**No es una corrección garantizada.** Puede ayudar con algunos términos, no cambiar nada, o
**inducir errores nuevos** si incluye palabras que no aparecen en el audio. Ese último caso
es el interesante: el modelo tiende a escribir lo que le dijiste que esperara.

In [ ]:
# Cambiá esto por términos que realmente aparezcan en TU audio.
GLOSARIO = (
    "Entrevista sobre procesamiento del lenguaje natural. "
    "Términos posibles: spaCy, tokenización, corpus, embeddings, "
    "TF-IDF, fastText, lematización."
)

print("Glosario de", len(GLOSARIO), "caracteres.")

La misma llamada de antes, con un parámetro más. Todo lo demás queda igual: es la
única forma de saber que lo que cambie se debe al glosario.

In [ ]:
generador_con_glosario, info_con_glosario = modelo.transcribe(
    audio=AUDIO,
    language="es",
    vad_filter=True,
    condition_on_previous_text=False,
    beam_size=5,
    initial_prompt=GLOSARIO,
)

print("Llamada hecha. Falta recorrer el generador.")

Y el recorrido. Acá solo guardamos el texto: los tiempos no cambian con el glosario,
así que no hace falta volver a anotarlos.

In [ ]:
segmentos_con_glosario = []

for segmento in generador_con_glosario:
    texto_limpio = segmento.text.strip()
    segmentos_con_glosario.append({"texto": texto_limpio})

cantidad_con_glosario = len(segmentos_con_glosario)
print("Segmentos con glosario:", cantidad_con_glosario)

Para comparar en serio hay que mirar los dos textos lado a lado, no uno después del
otro. La celda de abajo los pone en paralelo.

In [ ]:
cantidad_a_comparar = min(len(segmentos), len(segmentos_con_glosario))

for posicion in range(cantidad_a_comparar):
    if posicion >= 8:
        break
    sin_glosario = segmentos[posicion]["texto"]
    con_glosario = segmentos_con_glosario[posicion]["texto"]
    if sin_glosario != con_glosario:
        print("SIN:", sin_glosario)
        print("CON:", con_glosario)
        print()

### ✎ Para pensar

- ¿Cambió algo? Si no cambió nada, eso también es un resultado: quiere decir que el modelo
  ya resolvía bien esos términos.
- Si cambió, ¿mejoró o empeoró? Escuchá el audio en ese punto antes de contestar.
- Probá un glosario **con términos que no estén en el audio**. ¿Aparecen igual en la salida?

## ✎ Completá los pasos faltantes

```python
# ─── Comparar una configuración ──────────────────────────────────────────────
#
# Elegí UNA sola de estas tres cosas para cambiar, y dejá todo lo demás igual:
#     beam_size      probá con 1, y después con 10
#     vad_filter     probá con False
#     NOMBRE_MODELO  probá con "base"
#
# TODO 1: escribí abajo una llamada a modelo.transcribe() con tu cambio.
# TODO 2: guardá los segmentos en una lista llamada 'segmentos_de_la_prueba'.
# TODO 3: imprimí cuántos segmentos produjo y cuánto tardó.
# TODO 4: en markdown, escribí qué cambió y si te parece mejor o peor. Y por qué.
#
# ─────────────────────────────────────────────────────────────────────────────
```

## ✎ Espacio de práctica

```text
Consigna

Auditá esta transcripción como auditaste a spaCy en el cuaderno 04 de la unidad
anterior.

1. Elegí tres errores concretos de la salida y clasificalos: ¿es un problema del
   audio, del modelo, o del español rioplatense contra un modelo entrenado
   mayormente con otras variedades?
2. Para cada uno, escribí qué consecuencia tendría si este texto se usara sin
   revisión: en una cita textual, en un dato numérico, en un nombre propio.
3. Cerrá con una línea: ¿para qué usos alcanza esta transcripción tal como está,
   y para cuáles no?

Ninguno de estos errores hizo saltar una excepción. Todos devolvieron un resultado
con buena cara. Esa es la única cosa que hay que aprender acá.
```

## Qué sigue

Tenés el pipeline completo funcionando, pero solo funciona **para vos y dentro de este
cuaderno**. Alguien que no programa no puede usarlo: tendría que abrir Jupyter, entender
qué celda corre primero y editar una variable.

En el cuaderno 02 vamos a convertir esto en una herramienta con una interfaz, y a hacernos
la pregunta que aparece siempre que un pipeline se vuelve producto: qué mostramos, qué
escondemos y de qué advertimos.